In [25]:
# Download review data (text + rating). Uses stdlib only — works on macOS, Linux, and Colab.
import urllib.request

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Sports_and_Outdoors.jsonl.gz"
path = "real_reviews.jsonl.gz"
print(f"Downloading {path} ...")
urllib.request.urlretrieve(url, path)
print("Done.")


Done.


In [26]:
# Meta file (product titles). Filename matches later cells: reviews.jsonl.gz
import urllib.request

url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Sports_and_Outdoors.jsonl.gz"
path = "reviews.jsonl.gz"
print(f"Downloading {path} ...")
urllib.request.urlretrieve(url, path)
print("Done.")

Done.


In [23]:
# Installs into the *current Jupyter kernel* (same as python -m pip for that interpreter).
# Locally: pick your project .venv as the kernel to avoid Homebrew PEP 668 errors.
%pip install -q -U tqdm pandas


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [22]:
# Dependencies are installed in the cell above (%pip).
# Optional extras (uncomment if needed):
# %pip install -q -U requests beautifulsoup4 lxml html5lib

zsh:1: command not found: pip


In [27]:
import pandas as pd
import gzip
import json
from tqdm import tqdm

# The meta file you just downloaded
meta_file = 'reviews.jsonl.gz'
meta_mapping = {}

print("Extracting product titles from Meta file...")
with gzip.open(meta_file, 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        try:
            item = json.loads(line)
            # Map parent_asin to title
            meta_mapping[item['parent_asin']] = item.get('title', 'Unknown Product')
        except:
            continue
print(f"Successfully indexed {len(meta_mapping)} products.")

Extracting product titles from Meta file...


1587421it [00:34, 46348.94it/s]

Successfully indexed 1587421 products.


# Dataset Construction: Extracting the First 50,000 Reviews

This code constructs our initial dataset by extracting the first 50,000 reviews from the 2.45GB review file and enriching them with product names from the 1GB metadata file. It first builds a mapping dictionary linking each product ID (parent_asin) to its actual product name, then extracts the first 50,000 reviews and merges them with their corresponding product names. Finally, it adds sentiment labels (0 for negative, 1 for neutral, 2 for positive) and selects only the essential columns needed for analysis. The output is a lightweight CSV file containing 50,000 annotated reviews, ready for exploratory data analysis.

In [28]:
import pandas as pd
import gzip
import json
from tqdm import tqdm

# --- CONFIGURATION ---
# Ensure these filenames match exactly with the files in  Colab sidebar
meta_file = 'reviews.jsonl.gz'      # The 1GB Meta file
review_file = 'real_reviews.jsonl.gz' # The 2.45GB Review file
output_csv = 'sports_final_50k.csv'

meta_mapping = {}

# --- STEP 1: Building Product Title Mapping ---
print("Step 1: Building product title mapping from Meta file...")
with gzip.open(meta_file, 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        try:
            item = json.loads(line)
            # Map parent_asin to its real product title
            meta_mapping[item['parent_asin']] = item.get('title', 'Unknown Product')
        except:
            continue
print(f"Successfully indexed {len(meta_mapping)} products.")

# --- STEP 2: Extracting and Merging Reviews ---
data = []
print("\nStep 2: Extracting first 50,000 reviews and merging with titles...")
with gzip.open(review_file, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(tqdm(f, total=50000)):
        if i >= 50000:
            break
        try:
            review = json.loads(line)
            # Link the product name using the mapping from Step 1
            review['product_name'] = meta_mapping.get(review['parent_asin'], "Unknown Product")
            data.append(review)
        except:
            continue

# --- STEP 3: Transformation and Saving ---
df = pd.DataFrame(data)

# Sentiment Labeling (Based on your proposal: 1-2 stars=0, 3 stars=1, 4-5 stars=2)
df['sentiment'] = df['rating'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

# Selecting core columns for Modeling (RQ3) and Marketing Analysis (RQ2)
final_columns = ['product_name', 'text', 'rating', 'sentiment', 'parent_asin', 'verified_purchase']
final_df = df[final_columns]

# Save to a lightweight CSV (Only a few dozen MBs)
final_df.to_csv(output_csv, index=False)

print(f"\n✅ MISSION COMPLETE! Your final dataset '{output_csv}' has been saved.")
display(final_df.head())

Step 1: Building product title mapping from Meta file...


1587421it [00:34, 45919.24it/s]


Successfully indexed 1587421 products.

Step 2: Extracting first 50,000 reviews and merging with titles...


100%|██████████| 50000/50000 [00:00<00:00, 62108.12it/s]



✅ MISSION COMPLETE! Your final dataset 'sports_final_50k.csv' has been saved.


,product_name,text,rating,sentiment,parent_asin,verified_purchase
0,Alvada Merino Wool Hiking Socks Thermal Warm C...,Not gonna lie- they are not much to look at. L...,5.0,2,B0BGFR76CF,True
1,"CamelBak eddy Glass Water Bottle, 24oz",I love it. Pretty!,5.0,2,B00NXQLFQQ,True
2,B Vertigo Zurich Quilted Quick-Dry Dust-Resist...,Huge fan of B Vertigo and this dressage pad do...,5.0,2,B0957WLR63,True
3,"Weaver Leather Breakaway Fuse (3-Pack) , Brown","I have a great Weaver halter. Recently, the Ch...",5.0,2,B00IET8S80,True
4,Paris Tack English Leather Elastic Girth Exten...,This was great for a slightly too-short girth!...,5.0,2,B01C2SW7XA,True


# Balanced Dataset Construction: Under-Sampling for Class Balance

This code constructs a balanced dataset by selectively sampling reviews until reaching target quantities for each sentiment class: 15,000 negative (ratings 1-2), 15,000 neutral (rating 3), and 20,000 positive (ratings 4-5). It sequentially reads through the 2.45GB review file, evaluates each review's rating, and adds it to the corresponding class only if that class hasn't reached its target yet. The process stops immediately once all targets are met. This under-sampling technique ensures sufficient representation of the minority neutral class—critical for RQ3 analysis—while maintaining authentic review content without any text modification. The output is a 50,000-review balanced dataset saved as sports_balanced_50k.csv, ready for model training where class balance is essential.

In [ ]:
import gzip
import json
import pandas as pd
from tqdm import tqdm

# --- TARGET QUANTITIES ---
targets = {0: 15000, 1: 15000, 2: 20000}
counts = {0: 0, 1: 0, 2: 0}
balanced_data = []

review_file = 'real_reviews.jsonl.gz' # Using the 2.45GB file

print("Starting Balanced Extraction...")
with gzip.open(review_file, 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        # Stop if all targets are met
        if all(counts[k] >= targets[k] for k in targets):
            break

        try:
            review = json.loads(line)
            rating = review['rating']

            # Map rating to sentiment
            sentiment = 0 if rating <= 2 else (1 if rating == 3 else 2)

            # If we still need more of this sentiment, add it
            if counts[sentiment] < targets[sentiment]:
                balanced_data.append(review)
                counts[sentiment] += 1
        except:
            continue

# Create DataFrame
df_balanced = pd.DataFrame(balanced_data)
df_balanced['sentiment'] = df_balanced['rating'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

print("\nFinal Balanced Counts:")
print(df_balanced['sentiment'].value_counts())

# Save this as your NEW master file
df_balanced.to_csv('sports_balanced_50k.csv', index=False)

Starting Balanced Extraction...


209131it [00:03, 55155.10it/s]



Final Balanced Counts:
sentiment
2    20000
0    15000
1    15000
Name: count, dtype: int64


# Construction of Random Data Based on Reservoir Sampling



From the 2.63GB raw review file `real_reviews.jsonl.gz`, we randomly sampled 50,000 reviews using the reservoir sampling algorithm. This method requires only a single pass through the file and maintains a constant memory footprint of just 50,000 data points, while ensuring that every review has an exactly equal probability of being selected—thereby avoiding the sequential bias that might arise from simply selecting the first *N* entries. Following the sampling process, we matched each review with its corresponding product name using metadata files and filtered out low-quality reviews containing fewer than three words. This resulted in a final dataset of 46,361 high-quality entries, saved as `sports_random_50k_final.csv`, which will be used for subsequent analysis and model evaluation.

In [32]:
import gzip
import json
import pandas as pd
import random
import os
from tqdm import tqdm

# Output: Google Drive on Colab; current working directory locally (avoids /content read-only errors).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = '/content/drive/My Drive/6165 group'
except ImportError:
    save_dir = os.getcwd()
except OSError as e:
    print(f"Drive mount/path issue ({e}); using current directory.")
    save_dir = os.getcwd()

os.makedirs(save_dir, exist_ok=True)
print(f"Saving outputs under: {save_dir}")

review_file = 'real_reviews.jsonl.gz'
meta_file = 'reviews.jsonl.gz'

# 3. Build Product Meta Mapping (Titles)
meta_mapping = {}
print("\nStep 1: Building product name map...")
with gzip.open(meta_file, 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        try:
            data = json.loads(line)
            meta_mapping[data['parent_asin']] = data.get('title', 'Unknown Product')
        except:
            continue

# 4. Reservoir Sampling (50,000 Random Samples)
target_size = 50000
sampled_reviews = []
print(f"\nStep 2: Randomly sampling {target_size} reviews from millions...")
with gzip.open(review_file, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(tqdm(f)):
        if i < target_size:
            sampled_reviews.append(json.loads(line))
        else:
            r = random.randint(0, i)
            if r < target_size:
                sampled_reviews[r] = json.loads(line)

# 5. Data Processing & Quality Filtering
print("\nStep 3: Processing and filtering data...")
df_final = pd.DataFrame(sampled_reviews)
df_final['product_name'] = df_final['parent_asin'].map(meta_mapping)
df_final['sentiment'] = df_final['rating'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

# Filter: Remove reviews with fewer than 3 words (noise reduction)
df_final['word_count'] = df_final['text'].apply(lambda x: len(str(x).split()))
df_clean = df_final[df_final['word_count'] >= 3].copy()

# 6. Save Final Dataset
final_path = os.path.join(save_dir, 'sports_random_50k_final.csv')
df_clean.to_csv(final_path, index=False)

print(f"\n✅ SUCCESS! Randomized dataset saved at: {final_path}")
print(f"Final Count: {len(df_clean)} samples after filtering.")

# 7. Final Sanity Check (Distribution)
print("\n--- Final Sentiment Distribution (%) ---")
print(df_clean['sentiment'].value_counts(normalize=True) * 100)

Saving outputs under: /Users/jainesh/Downloads/Deep Learning/6165 group

Step 1: Building product name map...


1587421it [00:34, 45749.08it/s]



Step 2: Randomly sampling 50000 reviews from millions...


19595170it [00:46, 417676.10it/s]



Step 3: Processing and filtering data...

✅ SUCCESS! Randomized dataset saved at: /Users/jainesh/Downloads/Deep Learning/6165 group/sports_random_50k_final.csv
Final Count: 46393 samples after filtering.

--- Final Sentiment Distribution (%) ---
sentiment
2    77.990214
0    15.252301
1     6.757485
Name: proportion, dtype: float64
